# 13 · Change recipes: add, remove, modify, fix

Each recipe starts with `impact(...)`, the list of every file that mentions the thing you are
about to change, then names the edits and the tests. The examples are real changes made in this
repository.

In [ ]:
from nbkit import impact, pytest

## Recipe A · Add a Left source (e.g. a new database)

1. `backends/sources.py`: a `SqlSource` subclass with `dialect` and `connect()`; SQL generation,
   counting and pulling are inherited. Register it in `source_from_config`.
2. `cli/config.py`: add the name to `LEFT_SOURCES`.
3. `common/dates.py` (both copies) only if the dialect quotes identifiers or dates differently.
4. Optional driver → an extra in `pyproject.toml` and an `ImportError` naming it; update
   `test_packaging.py`'s extras list.
5. Tests: fake the driver module (`monkeypatch.setitem(sys.modules, ...)`) in `test_sources.py`;
   a container test in `tests/local/` if a Docker image exists.

This is how `FileSource` (DuckDB) was added.

In [ ]:
impact("source_from_config")

## Recipe B · Change a statistic or a normalisation rule

A statistic exists **three times**: pandas on the Left (`common/stats.py`), pandas for local Right
targets (the same module, other copy), Spark SQL in `SparkTarget.window_statistics`. The
comparison and the report consume its name.

1. Edit `common/stats.py` in both packages (parity test).
2. Edit the Spark SQL in `backends/targets.py` and, if the rule is textual, its Python twin test.
3. `checks/col_check.py::compare_statistics` if the comparison changes; report columns.
4. Tests: `tests/unit/common/test_stats.py`, `test_targets.py`, `just test spark`, live step 5.

`col_type_overrides` was added this way.

In [ ]:
impact("length_max")

## Recipe C · Change the protocol

Anything both processes read (object names, manifest fields, unit ids, pointer fields) is the
protocol. A Left of one version and a Right of another must not silently misread each other.

1. `common/manifest.py` (both copies): fields, `validate_manifest`, `PROTOCOL_VERSION` in both
   `__init__.py`.
2. Writers and readers on both sides.
3. Keep the ordering rules: payload before pointer, verdict last, pointer file last.
4. Tests: `test_manifest.py`, both `checks/` suites, `test_protocol_flow.py`, `just e2e`, live
   `test_job_flow.py`; notebooks 00 and 11 show the objects.

In [ ]:
impact("PROTOCOL_VERSION")

## Recipe D · Add a CLI command or option

1. Reuse the decorators in `cli/options.py` (`config_option`, `store_option`, `pair_option`,
   `log_level_option`, `emit_json`). Return a dict; `exit_code` in it becomes the exit status.
2. Put the command in the module of its kind (`checks.py` for pipeline steps, `tools.py` for
   helpers) and register it in `cli/__init__.py`. Mind the five-files-per-folder rule.
3. Anything external goes into `backends/`, taking its client as an argument so tests pass a fake.
4. Tests: wiring in `tests/unit/dtrack_left/cli/` (arguments reach the backend, JSON, exit
   codes 0/1/2); the backend in `tests/unit/dtrack_left/backends/`.

`dtrack dbx` followed exactly this split: `backends/databricks.py` + a thin group in `tools.py`.

In [ ]:
impact("DTRACK_DBX_")
pytest(
    "tests/unit/dtrack_left/cli/test_tools.py", "tests/unit/dtrack_left/backends/test_databricks.py"
)

## Recipe E · Remove a feature

Removing is the easiest place to leave dead ends: an option still documented, an env var still
in `.env.example`, a test still asserting the old header.

1. `impact("<name>")` over src, tests, docs and examples; delete every hit or rewrite it.
2. Remove parameters end to end, not only the CLI flag.
3. Replace tests that asserted the feature with tests that assert its absence where it matters.
4. Run the tiers that touched it, plus a live smoke if it reached the cloud.

KMS was removed this way. The table below is empty: nothing in src, tests, notes or notebooks
mentions it any more (only the historical design doc in `docs/plans/` does).

In [ ]:
impact("kms")

## Recipe F · Fix a bug that only a real service shows

1. Reproduce it in the smallest real tier (live, container) and keep the evidence.
2. Recreate the triggering condition in a **unit** test with a fake: this is the regression test.
3. Fix, run the unit tier, rerun the tier that found it.

Two bugs from this project, and the unit tests that now pin them:

In [ ]:
pytest(
    "tests/unit",
    "-k",
    "folder_marker or crash_before_the_manifest or differs_from_the_run_folder",
    "-v",
)

- **Folder markers**: Databricks writes a 0-byte `manifest/` object into a Volume-backed prefix;
  the test writes one with a raw `put_object`.
- **Verdict written too early**: a crash between verdict and manifest dropped a passed pair; the
  test makes `put_json("manifest/...")` raise and restarts.
- **Run id vs folder**: a mismatch between `--run-id` and the `--s3` folder is now a usage error.

## Before you open a pull request

```bash
just dev lint && just test unit && just test cov   # always
just test local && just e2e                        # when a real dependency is involved
just live test -m "not docker"                     # when Databricks or S3 behaviour changed
```
Then re-run `guardrails()` in notebook 12.